Attention has undoubtedly led to tremendous advances in recent years. However, this mechanism is computationally demanding, requiring quadratic time and space complexity. As a result, researchers proposed various approximations and alternatives in the early days when attention was still computed as a single operation, refered as "batch" attention in this post. 

With the growing necessity to compute attention faster and over longer context lengths, a simple trick has far surpassed all other attempts at optimization: computing attention in an online fashion. I will call this as "online" attention. This approach processes attention piecewise, using only portions of keys, values, and queries at any given moment. It's both simple and accurate, involving no approximations.

Additionally, the online nature of attention may lead to more stable results, as the impact of larger exponents is felt only through accumulated sums. In contrast, batch attention computation can cause logits with low values (below -11.09 for FP8, when exp(-11.09) becomes smaller than the lowest representable number) to immediately drop to zero, potentially losing the contribution of corresponding value vectors. Whether this is beneficial or detrimental remains an open question, if you have insights on this, I'd love to hear them!


In this post, I will try to deconstruct how attention is computed by reading keys, values, and queries in a streaming fashion. 
To avoid memory overheads from storing intermediate representations such as score matrix we will also derive gradients through attention in an online fashion, i.e., by reading only parts of keys, values, and queries. 
I will be using code to demonstrate these concepts in practice. 

This is the mechanism behing [FlashAttention](https://github.com/Dao-AILab/flash-attention) (computing attention on a single GPU core by fusing operations and managing memory in a clever way) as well as [RingAttention](https://arxiv.org/abs/2310.01889), computing attention by cleverly overlapping partial computation of attention with communication of keys and values across GPUs. 

## Forward Pass

We have $Q \in \mathcal{R}^{N \times d}$, $K \in \mathcal{R}^{M \times d}$, and $V \in \mathcal{R}^{M \times d}$.

We want to compute 
$$
\text{Attn}(Q, K, V) = \text{Softmax}(\tau QK^T)V
$$

Let's break it down 

$$
S = QK^T \\
P = \text{Softmax}(\tau S) \\
O = PV
$$


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [129]:
B = 10
L = 20
D = 16
Q = torch.randn(B, L, D, requires_grad=True)
K = torch.randn(B, L, D, requires_grad=True)
V = torch.randn(B, L, D, requires_grad=True)

tau = 1. / math.sqrt(D)

# Using PyTorch Module
# API expects [B, heads, L, D]; here head dim = 1
O = F.scaled_dot_product_attention(Q.unsqueeze(1), K.unsqueeze(1), V.unsqueeze(1))
O = O.squeeze(1)

# Handcoded batch attention
scores = tau * torch.matmul(Q, K.transpose(-2, -1))
sm = torch.softmax(scores, dim=-1)
O_manual = torch.matmul(sm, V)

print("O == O_manual: ", torch.allclose(O, O_manual, atol=1e-6))


O == O_manual:  True


## Forward pass in online fashion

We want to compute the above in online fashion, where only the blocks of Q, K, and V are available at a time. This might be due to memory constraints of GPU (e.g., FlashAttention) or due to the harware memoery constraints (e.g., Ring Attention). 

Now, note that the full computation can be written as 

$$
S_{ij} = Q_i^TK_j \\ 

P_{ij} = \frac{\exp (\tau S_{ij})}{\sum_{j} \exp (\tau S_{ij})} \\

O_{i, :} = \sum_{j}P_{i, j} \times V_{j, :}
$$

Thus, we want to compute 

$$
O_{i, :} = \frac{\sum_{j}\exp (\tau Q_i^TK_j)  \times V_{j, :}}{\sum_{j} \exp (\tau Q_i^TK_j)}
$$

To avoid overflow resulting from large exponents, we normalize the expoenentials by subtracting the maximum of scores from their exponents. 

$$
m_i = \max_{j}Q_i^TK_j \\

O_{i, :} = \frac{\sum_{j}\exp (\tau Q_i^TK_j - m_i)  \times V_{j, :}}{\sum_{j} \exp (\tau Q_i^TK_j  - m_i)}
$$

In [130]:
tau = 1. / math.sqrt(Q.shape[-1])
scores = torch.matmul(Q, K.transpose(-2, -1))
max_per_query = scores.max(dim=-1, keepdims=True).values
A = torch.exp(tau * (scores - max_per_query))
A = A / torch.sum(A, axis=-1, keepdims=True)
O_m_with_max = torch.matmul(A, V)

print("O == O_m_with_max: ", torch.allclose(O, O_m_with_max, atol=1e-6))

O == O_m_with_max:  True


But, we want to do it in the online fashion, where we see blocks of queries, keys, and values, let's do it now:

Let's denote the block of queries by $B_q$, and the block of keys and values as $B_k$.
Thus, we have $Q_{B_q} \in \mathcal{R}^{B_q \times d}$, and $K_{B_k} \in \mathcal{R}^{B_k \times d}$, and $V \in \mathcal{R}^{B_k \times d}$, where $B_q < N$ and $B_k < M$.
To do the above compuation, when only blocks of $Q, K$, and $V$ are available, we need to break it down in blocks. 


For convenience, imagine $B_q$ and $B_k$ to be $1$. For a given query, when we have read the $t$-th key and value, we can upadte the numerator, denominator, and m_i as follows:

$$
m_i^t = \max (m_i^{t-1}, Q_i^TK_t) \in \mathcal{R}\\ 

n_i^t = n_i^{t-1} \times \exp (m_i^{t-1} - m_i^t) + \exp (\tau Q_i^TK_t - m_i^{t}) \times V_t \in \mathcal{R}^d \\ 

d_i^t = d_i^{t-1} \times \exp (m_i^{t-1} - m_i^t) + \exp (\tau Q_i^TK_t - m_i^{t}) \in \mathcal{R} \\

$$

Note that the correction term $\exp (m_i^{t-1} - m_i^t)$ accounts for the update in the maximum for normalizing the exponents. 

At the end of the $M$-th reading of the keys, we will have

$$
O_{i, :} = \frac{n_i^{M}}{d_i^{M}}
$$

Thus, we can compute the attention in an online fashion, where we only need to keep the blocks of Query, Keys, and Values in memory and update the output matrix iteratively.

The algorithm is almost same except for that we keep an account of max, numerator and denominator as we see the stream of Q,K, V blocks. 


In [132]:
block_size = 2
tau = 1. / math.sqrt(Q.shape[-1])

max_per_query = torch.full((B, L), float('-inf'), dtype=torch.float32) # one max per query
num = torch.zeros(B, L, D)
den = torch.zeros(B, L)

for start_idx in range(0, L, block_size):
    outer_range = range(start_idx, min(start_idx + block_size, L))
    K_b, V_b = K[:, outer_range], V[:, outer_range] # [B, BLOCK, D]

    for start_jdx in range(0, L, block_size):
        inner_range = range(start_jdx, min(start_jdx + block_size, L))
        Q_b = Q[:, inner_range]

        scores_b = tau * torch.matmul(Q_b, K_b.transpose(-2, -1))

        ## Bookkeeping
        m_old = max_per_query[:, inner_range]

        # compute new maximums
        m_new = max_per_query[:, inner_range] = torch.max(m_old, scores_b.max(axis=-1).values)
        
        # rescaling factors
        exp_delta_m = torch.exp(m_old - m_new).unsqueeze(-1)
        new_score_exp = torch.exp(scores_b  - m_new.unsqueeze(-1)) # Broadcast max to the last dimension

        # numerator
        num[:, inner_range] = num[:, inner_range] * exp_delta_m + torch.matmul(new_score_exp, V_b)

        # denominator 
        den[:, inner_range] = den[:, inner_range] * exp_delta_m.squeeze(-1) + new_score_exp.sum(dim=-1)

        
O_online = num/den.unsqueeze(-1)

print("O == O_online: ", torch.allclose(O, O_online, atol=1e-6))

O == O_online:  True


## Backward Pass

Let's simplify the above notations a bit. At the end of the above forward computation, we have access to the sum in the denominator, $d_i = d^{M}_i$, and the $m_i = \max_j Q_i^TK_j$. 

During the backward pass, we are interested in computing $\frac{\partial L}{\partial Q}$, $\frac{\partial L}{\partial K}$, $\frac{\partial L}{\partial V}$, given $\frac{\partial L}{\partial O}$, where $L \in \mathcal{R}$ is the loss function. For simplicity, we will avoid typing $\partial L$ so that $\partial O = \frac{\partial L}{\partial O}$ and so on. 

Note,
$$
\partial O \in \mathcal{R}^{N \times d}, \quad \partial Q \in \mathcal{R}^{N \times d}, \quad  \partial K \in \mathcal{R}^{M \times d}, \quad  \partial V \in \mathcal{R}^{M \times d}
$$

$$
O = PV  \in \mathcal{R}^{N \times d}\\

\partial V = P^T \partial O \in \mathcal{R}^{M \times d}, \qquad \partial P = \partial O V^T \in \mathcal{R}^{N \times M} \\
$$

Since each $P_{i, :}$ is a result of the softmax on the $i$-th row of $S$, while taking the derivative, we need to treat them together. 

Specifically,

$$
\mathbf{y} = \text{Softmax}(\mathbf{x}) \in \mathcal{R}^d \\

\partial \mathbf{x} = (\text{diag}(\mathbf{y}) - \mathbf{y}\mathbf{y}^T) \times \partial \mathbf{y}
$$

Above follows from the following observation - 

$$
\frac{\partial \mathbf{y}_{i}}{\partial \mathbf{x}_j} = \sigma_i \sigma_j, \quad \text{if i} \neq {j} \\
= \sigma_i (1 - \sigma_i), \text{otherwise}
$$

Applying it to the $i$-th row of $P$, we have

$$
\partial S_{i, :} = (\text{diag}(\mathbf{P_{i, :}}) - \mathbf{P_{i, :}}\mathbf{P_{i, :}}^T) \times \partial \mathbf{P_{i, :}} \in \mathcal{R}^{M},
$$

$$
\partial S_{i, :} =  \mathbf{P_{i, :}} \odot (\partial \mathbf{P_{i, :}} - (\partial \mathbf{P_{i, :}}^T\mathbf{P_{i, :}})\mathbf{1})
$$

where I have assumed $\mathbf{P_{i, :}} \in \mathcal{R}^M$, a column vector even though it represents a row in the matrix $P$ 

Given $\partial S_{ij}$, we can proceed in the similar fashion to compute $\partial Q$ and $\partial K$ since $S = QK^T$ is the matrix multiplication.

$$
\partial Q = \partial S K \in \mathcal{R}^{N \times d}, \qquad \partial K^T = Q^T \partial S \in \mathcal{R}^{d \times M}

$$


If you are an expert in matrix calculus, above is easy. If you are not, checkout the appendi at the end on how to arrive at the above formulas. 

In [133]:
# simulating gradient wrt the loss
dO = torch.randn(B, L, D)
O.backward(dO) # perform backprop

# read off the gradients 
dQ = Q.grad 
dK = K.grad
dV = V.grad 

# # Handcoded batch attention
scores = tau * torch.matmul(Q, K.transpose(-2, -1))
P = torch.softmax(scores, dim=-1)
O_manual = torch.matmul(P, V)

print("O == O_manual: ", torch.allclose(O, O_manual, atol=1e-6))

# backward pass in batch updates
dV_manual = torch.matmul(sm.transpose(-2, -1), dO)

dP = torch.matmul(dO, V.transpose(-2, -1))
dScores = P * (dP -  (dP * P).sum(dim=-1).unsqueeze(-1))

dQ_manual = tau * torch.matmul(dScores, K) # multiply the scaling factor 
dK_manual = tau * torch.matmul(dScores.transpose(-2, -1), Q) # multiply the scaling factor 

print("dV == dV_manual: ", torch.allclose(dV, dV_manual, atol=1e-6))
print("dQ == dQ_manual: ", torch.allclose(dQ, dQ_manual, atol=1e-6))
print("dK == dK_manual: ", torch.allclose(dK, dK_manual, atol=1e-6))

O == O_manual:  True
dV == dV_manual:  True
dQ == dQ_manual:  True
dK == dK_manual:  True


## Backward pass in online fashion 

Notice that computing the gradients in the straightforward way would require storing the full softmax matrix sm in batch form. For long sequences, this quickly becomes impractical. For example, with a sequence length of 128K, the attention matrix has size 
$128𝐾 \times 128𝐾$. At 32-bit precision, that means storing around 65 GB just for one example. Clearly, this is not feasible.

Here, we work through the computation that FlashAttention does.

We need to recompute scores in our backward pass. 
We follow the same pattern as in the forward pass, where we load blocks of query, keys, and values.
The difference being, we will be computing the scores and softmax in again, except that we already have their maxes so we don't need to do any bookkeeping. Same with the denomointor sum as that is also what we store.

But before we proceed, we need to take care of the scalar dot product that we used in computing the derivatives with respect to the scores.
Specifically, $\partial P_{i:}^T P_{i:}$ computes the scalar dot product that is broadcasted to all the components of $P$. 
This prevents parallelizing it naively. 

As a result, FlashAttention paper uses the following simplification:

$$
D_i = \partial P_{i:}^T P_{i:} = \sum_{j}P_{ij}\partial P_{ij} = \sum_j P_{ij} \partial O_i V_j = \partial O_i \sum_j P_{ij} V_j = \partial O_i^TO_i
$$

In the above, I have row-colum view of matrix multiplication,

$$
\partial P = \partial O V^T,
\partial P_{ij} = \partial O_i V_j
$$


In [150]:
block_size = 2
tau = 1. / math.sqrt(Q.shape[-1])

# given:
# max_per_query: max over the query-key inner product
# den: denominator sum exp (query, key inner product) per query


dQ_online = torch.zeros(B, L ,D)
dK_online = torch.zeros(B, L ,D)
dV_online = torch.zeros(B, L ,D)

for start_idx in range(0, L, block_size):
    outer_range = range(start_idx, min(start_idx + block_size, L))
    K_b, V_b = K[:, outer_range], V[:, outer_range] # [B, BLOCK, D]

    for start_jdx in range(0, L, block_size):
        inner_range = range(start_jdx, min(start_jdx + block_size, L))
        Q_b = Q[:, inner_range]
        dO_b = dO[:, inner_range]
        O_b = O[:, inner_range]

        # recompute local probs P_b exactly using global m, d for the query rows
        scores_b = tau * torch.matmul(Q_b, K_b.transpose(-2, -1)) 
        m_b = max_per_query[:, inner_range, None]
        d_b = den[:, inner_range, None] 
        P_b = torch.exp(scores_b - m_b) / d_b # these are exact attn weights 
        
        # computing the derivatives wrt V and P
        dV_b = torch.matmul(P_b.transpose(-2, -1), dO_b)
        dP_b = torch.matmul(dO_b, V_b.transpose(-2, -1))

        # computing the derivatives wrt scores
        D_i = (dO_b * O_b).sum(dim=-1)
        dScores_b = P_b * (dP_b -  D_i.unsqueeze(-1))

        # computing the derivatives wrt Q and K
        dQ_b = tau * torch.matmul(dScores_b, K_b) # multiply the scaling factor 
        dK_b = tau * torch.matmul(dScores_b.transpose(-2, -1), Q_b) # multiply the scaling factor 

        # accumulating these gradients
        dQ_online[:, inner_range] += dQ_b
        dK_online[:, outer_range] += dK_b
        dV_online[:, outer_range] += dV_b
        
print("dV == dV_online: ", torch.allclose(dV, dV_online, atol=1e-6))
print("dQ == dQ_online: ", torch.allclose(dQ, dQ_online, atol=1e-6))
print("dK == dK_online: ", torch.allclose(dK, dK_online, atol=1e-6))

dV == dV_online:  True
dQ == dQ_online:  True
dK == dK_online:  True


## Appendix A: Full Attention with Dropout and Mask

Now, that we are familiar with how to compute the forward and backward in online fashion, lets' see the full pass along with dropout and masking. 
It will be almost same, except for the parts to handle dropout and masking both in the forward and the backward passes. 

Below is a rough pass that includes both.

The change in the online backward pass will be to generate the scores, apply the mask and dropout  for every block. This requires extra-careful handling of the random number generator so that the forward and the backward pass rematerialization does the same dropouts. 
I will exclude that from here as I myself don;t know how to do it. 

In [182]:
B = 10
L = 20
D = 16
dropout_p = 0.
Q = torch.randn(B, L, D, requires_grad=True)
K = torch.randn(B, L, D, requires_grad=True)
V = torch.randn(B, L, D, requires_grad=True)
MASK = torch.ones(L, L, dtype=bool).tril(diagonal=0) # 0 offset from the diagonal

tau = 1. / math.sqrt(D)

# Using PyTorch Module
# API expects [B, heads, L, D]; here head dim = 1
O = F.scaled_dot_product_attention(Q.unsqueeze(1), K.unsqueeze(1), V.unsqueeze(1), attn_mask=MASK, dropout_p=dropout_p)
O = O.squeeze(1)

# Handcoded batch attention
scores = tau * torch.matmul(Q, K.transpose(-2, -1))
scores.masked_fill_(~MASK, float('-inf')) # apply mask
P = torch.softmax(scores, dim=-1)

# dropout on some attention weights 
keep_prob = 1 - dropout_p
dropout_mask = (torch.rand_like(P) < keep_prob).to(sm.dtype)
P = P * dropout_mask / keep_prob # We divide by keep_prob so that we don't have to multiply this factor at the test time

O_manual = torch.matmul(P, V)
print(f"O == O_manual (True if dropout_p = 0.0, current dropout: {dropout_p}): ", torch.allclose(O, O_manual, atol=1e-6))


## Computing backwards

# simulate gradients of loss wrt O
dO = torch.rand_like(O)
O.backward(dO)
dQ = Q.grad 
dK = K.grad 
dV = V.grad

dV_manual = torch.matmul(P.transpose(-2, -1), dO)
dP = torch.matmul(dO, V.transpose(-2, -1))
dP = dP * dropout_mask / keep_prob
dScores = P * (dP - (dP * P).sum(dim=-1).unsqueeze(-1))
dQ_manual = tau * torch.matmul(dScores, K) # multiply the scaling factor 
dK_manual = tau * torch.matmul(dScores.transpose(-2, -1), Q) # multiply the scaling factor 

print("dV == dV_manual: ", torch.allclose(dV, dV_manual, atol=1e-6))
print("dQ == dQ_manual: ", torch.allclose(dQ, dQ_manual, atol=1e-6))
print("dK == dK_manual: ", torch.allclose(dK, dK_manual, atol=1e-6))


O == O_manual (True if dropout_p = 0.0, current dropout: 0.0):  True
dV == dV_manual:  True
dQ == dQ_manual:  True
dK == dK_manual:  True


# Appendix B: Computing derivatives of P and V

Here we will build the intuition to perform the following,

$$
O = PV  \in \mathcal{R}^{N \times d}\\

\partial V = P^T \partial O \in \mathcal{R}^{M \times d}, \qquad \partial P = \partial O V^T \in \mathcal{R}^{N \times M} \\
$$


## Matrix Calculus View

Let’s start with some definitions. In calculus, the **gradient** is just the derivative, and **differentials** are small changes in variables, also called **variations**.

Our goal is to understand how to get the gradients of the loss with respect to $P$ and $V$, i.e. $\nabla_P L$ and $\nabla_V L$, when we already know the gradient with respect to $O$, denoted $\nabla_O L$. Here the relation is

$$
O = P V, \quad O, P, V \in \mathbb{R}^{N \times D}.
$$


#### Step 1. Variations of $O$

The first step is to ask: if $P$ and $V$ change a little bit, how does $O$ change? Denote these small changes by $\delta P$, $\delta V$, and $\delta O$.

For the function $f(P,V) = P V$, the change in output is

$$
\delta O = P \,\delta V + \delta P \,V.
$$

This is just the usual product rule, written in matrix form: wiggle $P$, you get $\delta P V$; wiggle $V$, you get $P \delta V$.


#### Step 2. Variation of the loss

Now we want to see how the loss $L$ changes when $O$ changes. By definition, the variation in $L$ is given by the inner product between the gradient and the perturbation:

$$
\delta L = \mathrm{tr} \!\left( (\nabla_O L)^\top \delta O \right).
$$

Substitute $\delta O = P \delta V + \delta P V$:

$$
\delta L
= \mathrm{tr}\!\big((\nabla_O L)^\top P \,\delta V\big)
+ \mathrm{tr}\!\big((\nabla_O L)^\top \delta P V\big).
$$


#### Step 3. Rearranging with the trace identity

Using the identity $\mathrm{tr}(ABC) = \mathrm{tr}(CBA)$, we can move terms around so that $\delta V$ and $\delta P$ appear at the end:

$$
\delta L
= \mathrm{tr}\!\big((P^\top \nabla_O L)^\top \delta V\big)
+ \mathrm{tr}\!\big((\nabla_O L V^\top)^\top \delta P\big).
$$

Now it’s clear which matrices multiply with $\delta V$ and $\delta P$.


#### Step 4. Read off the gradients

By definition of the trace/Frobenius inner product, the coefficients of $\delta V$ and $\delta P$ are the gradients we seek:

$$
\nabla_V L = P^\top \nabla_O L,
\qquad
\nabla_P L = \nabla_O L V^\top.
$$


## Row-wise perturbation view 

### Computing $\nabla_V L$

Let’s focus on the $j$-th row of $V \in \mathbb{R}^{M \times d}$.

From the forward computation (row form of matrix multiplication), we have

$$
O_{i,:} = \sum_{j=1}^M P_{ij}\, V_{j,:}.
$$

So each row $V_{j,:}$ contributes to the output row $O_{i,:}$ with weight $P_{ij}$.


#### Step 1. Effect of perturbing $V_{j,:}$

If we perturb $V_{j,:}$ by $\delta V_{j,:}$, then the corresponding change in $O_{i,:}$ is

$$
\delta O_{i,:} = P_{ij}\,\delta V_{j,:}, \qquad \forall i \in \{1,\dots,N\}.
$$

#### Step 2. Effect on the loss

How do we connect the change in $O$ to the change in the loss $L$?
By definition, the gradient of $L$ with respect to $O_{i,:}$ tells us how sensitive the loss is to small changes in that row. The precise rule is:

$$
\delta L = \sum_{i=1}^N \Big\langle \nabla_{O_{i,:}} L,\, \delta O_{i,:} \Big\rangle.
$$

Here $\langle \cdot , \cdot \rangle$ is just the standard inner product on row vectors.
This says: *the change in the loss is the dot product between the gradient and the perturbation, summed over all rows.*

Now plug in the perturbation $\delta O_{i,:} = P_{ij}\,\delta V_{j,:}$:

$$
\delta L = \sum_{i=1}^N \Big\langle \nabla_{O_{i,:}} L,\, P_{ij}\,\delta V_{j,:} \Big\rangle.
$$

Factor out the common $\delta V_{j,:}$:

$$
\delta L = \Big\langle \sum_{i=1}^N P_{ij}\,\nabla_{O_{i,:}} L,\, \delta V_{j,:} \Big\rangle.
$$


#### Step 3. Read off the gradient

From the definition of the inner product, this gives

$$
\frac{\partial L}{\partial V_{j,:}} = \sum_{i=1}^N P_{ij}\,\frac{\partial L}{\partial O_{i,:}}.
$$

Stacking over all rows $j$, we get the compact matrix form:

$$
\nabla_V L = P^\top \nabla_O L.
$$

### Computing $\nabla_{P_{ij}} L$

Let’s now switch to the **column view** of matrix multiplication. Each entry of $P$ picks out a column of $V$ and scales it:

$$
O_{i,:} = \sum_{j=1}^M P_{ij}\, V_{j,:}.
$$

So, if we perturb just a single entry $P_{ij}$ by a small amount $\delta P_{ij}$, the corresponding perturbation in the output row is

$$
\delta O_{i,:} = \delta P_{ij}\, V_{j,:}.
$$


#### Step 1. Effect on the loss

By definition, the change in the loss is given by the inner product between the gradient and the perturbation:

$$
\delta L = \big\langle \nabla_{O_{i,:}} L,\; \delta O_{i,:} \big\rangle.
$$

Substitute $\delta O_{i,:} = \delta P_{ij}\,V_{j,:}$:

$$
\delta L = \big\langle \nabla_{O_{i,:}} L,\; \delta P_{ij}\, V_{j,:} \big\rangle.
$$


#### Step 2. Factor out the scalar $\delta P_{ij}$

Since $\delta P_{ij}$ is just a number, we can pull it out:

$$
\delta L = \big(\nabla_{O_{i,:}} L^\top V_{j,:}\big)\,\delta P_{ij}.
$$


#### Step 3. Read off the gradient

From the definition of the inner product, the coefficient multiplying $\delta P_{ij}$ is exactly the gradient:

$$
\nabla_{P_{ij}} L = \nabla_{O_{i,:}} L^\top V_{j,:}.
$$


#### Step 4. Collect into matrix form

Stacking all entries together gives the compact expression:

$$
\nabla_P L = \nabla_O L\, V^\top.
$$

